# Flipkart Gridlock Hackathon 2.0: Traffic Demand Prediction
### **Submission Folder: State-of-the-Art ML Predictive Pipeline**

---

## 1. Executive Summary & Project Objective

Traffic congestion is a multi-billion dollar hurdle in modern urban logistics, impacting delivery efficiency, resource allocation, and carbon footprints. Predictively understanding local travel and booking demand patterns allows fleet managers to proactively deploy drivers, optimize warehouse stock pre-staging, and implement dynamic pricing mechanisms.

**Objective:** Build a highly accurate and generalizable machine learning model to predict the traffic/travel demand (`demand`) at a given location (`geohash`) and time (`timestamp`, `day`), utilizing temporal, road-type, and environmental data. 

**Our Winning Strategy:**
1. **Spatial Feature Engineering:** Decoding geographic `geohash` strings into raw latitude/longitude coordinates and running unsupervised spatial K-Means clustering to create cohesive geographic hubs.
2. **Cyclic Temporal Encoding:** Mapping times and days to multi-dimensional trigonometric vectors ($\sin$, $\cos$) so the model naturally captures continuous diurnal and weekly patterns.
3. **Out-of-Fold Target Encoding:** Engineering leakage-free historical baseline statistics of demand for locations.
4. **Optimized Ensemble:** Building and training a combined multi-model framework utilizing **LightGBM**, **XGBoost**, and **CatBoost** regressors using a **5-Fold Cross-Validation** strategy.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

sns.set_theme(style="darkgrid", palette="muted")
warnings.filterwarnings('ignore')
print("Libraries loaded successfully!")

## 2. Advanced Feature Engineering

### Spatial Geohash Decoding to Coordinates
A major limitation of standard tabular splits is that a categorical `geohash` string has high cardinality, and standard encodings hide spatial relationships. We convert geohashes back to Latitude and Longitude to calculate spatial proximity and cluster location behaviors.

In [ ]:
def decode_geohash(geohash):
    if not isinstance(geohash, str) or len(geohash) == 0:
        return np.nan, np.nan
    
    BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"
    char_to_val = {char: i for i, char in enumerate(BASE32)}
    
    lat_interval = [-90.0, 90.0]
    lon_interval = [-180.0, 180.0]
    
    is_even = True
    for char in geohash.lower():
        if char not in char_to_val:
            return np.nan, np.nan
        val = char_to_val[char]
        for mask in [16, 8, 4, 2, 1]:
            bit = 1 if (val & mask) else 0
            if is_even:
                mid = (lon_interval[0] + lon_interval[1]) / 2.0
                if bit == 1:
                    lon_interval[0] = mid
                else:
                    lon_interval[1] = mid
            else:
                mid = (lat_interval[0] + lat_interval[1]) / 2.0
                if bit == 1:
                    lat_interval[0] = mid
                else:
                    lat_interval[1] = mid
            is_even = not is_even
            
    lat = (lat_interval[0] + lat_interval[1]) / 2.0
    lon = (lon_interval[0] + lon_interval[1]) / 2.0
    return lat, lon

### Loading Data and Building the Feature Suite
We load training and test sets and apply a clean, uniform pipeline.

In [ ]:
data_dir = "data/dataset"
train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))

test_idx = test['Index'].copy()
df = pd.concat([train.drop(columns=['demand'], errors='ignore'), test], ignore_index=True)

# 1. Decode geohashes to lat/long coordinates
coords = df['geohash'].apply(decode_geohash)
df['latitude'] = [c[0] for c in coords]
df['longitude'] = [c[1] for c in coords]
df['latitude'] = df['latitude'].fillna(df['latitude'].mean())
df['longitude'] = df['longitude'].fillna(df['longitude'].mean())

# 2. K-Means clustering for location grouping
kmeans = KMeans(n_clusters=15, random_state=42, n_init=10)
df['spatial_cluster'] = kmeans.fit_predict(df[['latitude', 'longitude']])

# 3. Time features
def parse_time(t_str):
    if not isinstance(t_str, str) or ':' not in t_str:
        return 12.0
    h, m = map(int, t_str.split(':'))
    return h + m / 60.0

df['hour'] = df['timestamp'].apply(parse_time)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
df['day_of_week'] = df['day'] % 7
df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7.0)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7.0)
df['is_rush_hour'] = (((df['hour'] >= 8.0) & (df['hour'] <= 10.0)) | 
                      ((df['hour'] >= 17.0) & (df['hour'] <= 20.0))).astype(int)
df['is_night'] = ((df['hour'] >= 22.0) | (df['hour'] <= 5.0)).astype(int)

# 4. Handling missing entries & categorical mapping
df['RoadType'] = df['RoadType'].fillna('Unknown')
df['Weather'] = df['Weather'].fillna('Unknown')
df['LargeVehicles'] = df['LargeVehicles'].fillna('Not Allowed').map({'Allowed': 1, 'Not Allowed': 0})
df['Landmarks'] = df['Landmarks'].fillna('No').map({'Yes': 1, 'No': 0})
df['Temperature'] = df['Temperature'].fillna(df.groupby(['day', 'Weather'])['Temperature'].transform('median'))
df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
df['NumberofLanes'] = df['NumberofLanes'].fillna(df['NumberofLanes'].median())

cat_cols = ['RoadType', 'Weather', 'geohash']
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

df['temp_lanes_interaction'] = df['Temperature'] * df['NumberofLanes']

train_feats = df.iloc[:len(train)].copy()
test_feats = df.iloc[len(train):].copy()
train_feats['demand'] = train['demand'].values

print("Preprocessing completed successfully!")

### Out-of-Fold Spatial Target Encoding
We calculate the historical average demand per location in a clean, leakage-free way during validation folds, utilizing global training fallback mapping for the test set.

In [ ]:
train_feats['geohash_mean_demand'] = np.nan
test_feats['geohash_mean_demand'] = np.nan

kf = KFold(n_splits=5, shuffle=True, random_state=42)
global_geohash_mean = train_feats.groupby('geohash')['demand'].mean()
global_cluster_mean = train_feats.groupby('spatial_cluster')['demand'].mean()
global_overall_mean = train_feats['demand'].mean()

for train_idx, val_idx in kf.split(train_feats):
    fold_train = train_feats.iloc[train_idx]
    fold_geohash_mean = fold_train.groupby('geohash')['demand'].mean()
    fold_cluster_mean = fold_train.groupby('spatial_cluster')['demand'].mean()
    
    train_feats.iloc[val_idx, train_feats.columns.get_loc('geohash_mean_demand')] = \
        train_feats.iloc[val_idx]['geohash'].map(fold_geohash_mean).fillna(
            train_feats.iloc[val_idx]['spatial_cluster'].map(fold_cluster_mean)
        ).fillna(global_overall_mean)
        
test_feats['geohash_mean_demand'] = test_feats['geohash'].map(global_geohash_mean).fillna(
    test_feats['spatial_cluster'].map(global_cluster_mean)
).fillna(global_overall_mean)

features_to_drop = ['Index', 'timestamp']
X = train_feats.drop(columns=['demand'] + features_to_drop, errors='ignore')
y = train_feats['demand']
X_test = test_feats.drop(columns=features_to_drop, errors='ignore')
print("Spatial out-of-fold mapping completed!")

## 3. High-Performance Multi-Model GBDT Ensemble

We train our robust ensemble using **5-Fold Cross-Validation**. Out-of-fold forecasts are collected for **LightGBM**, **XGBoost**, and **CatBoost** regressors and blended optimally.

In [ ]:
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

preds_lgb = np.zeros(len(X_test))
preds_xgb = np.zeros(len(X_test))
preds_cat = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"=== Fold {fold+1} ===")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # 1. LightGBM
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    lgb_params = {
        'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.05,
        'max_depth': 8, 'num_leaves': 64, 'verbose': -1, 'random_state': 42
    }
    model_lgb = lgb.train(lgb_params, train_data, num_boost_round=1200, valid_sets=[train_data, val_data],
                          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    oof_lgb[val_idx] = model_lgb.predict(X_val, num_iteration=model_lgb.best_iteration)
    preds_lgb += model_lgb.predict(X_test, num_iteration=model_lgb.best_iteration) / 5
    
    # 2. XGBoost
    model_xgb = xgb.XGBRegressor(n_estimators=1200, learning_rate=0.05, max_depth=7, subsample=0.8, colsample_bytree=0.8,
                                 random_state=42, early_stopping_rounds=50, eval_metric='rmse')
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict(X_val)
    preds_xgb += model_xgb.predict(X_test) / 5
    
    # 3. CatBoost
    model_cat = CatBoostRegressor(iterations=1200, learning_rate=0.05, depth=7, eval_metric='RMSE', random_seed=42,
                                  early_stopping_rounds=50, verbose=False)
    model_cat.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
    oof_cat[val_idx] = model_cat.predict(X_val)
    preds_cat += model_cat.predict(X_test) / 5

### Validation Metrics & Weighted Blending Search
We evaluate the out-of-fold $R^2$ score for each model and perform a grid search to discover the optimal blending weights.

In [ ]:
score_lgb = r2_score(y, oof_lgb)
score_xgb = r2_score(y, oof_xgb)
score_cat = r2_score(y, oof_cat)

print(f"LightGBM R2: {score_lgb*100:.4f}%")
print(f"XGBoost  R2: {score_xgb*100:.4f}%")
print(f"CatBoost R2: {score_cat*100:.4f}%")

best_score = 0
best_w = (0.33, 0.33, 0.34)
for w1 in np.linspace(0, 1, 21):
    for w2 in np.linspace(0, 1-w1, 21):
        w3 = 1.0 - w1 - w2
        blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
        score = r2_score(y, blend)
        if score > best_score:
            best_score = score
            best_w = (w1, w2, w3)
            
print(f"Optimal weights: LGB: {best_w[0]:.2f}, XGB: {best_w[1]:.2f}, CAT: {best_w[2]:.2f}")
print(f"Ensemble Blended R2 Score: {best_score*100:.4f}%")

## 4. Model Explainability: Feature Importance

A key element in deploying machine learning to real-world logistics is **explainability**. We plot the feature importance of our models to show exactly which variables are driving the predictions.

In [ ]:
# Plot LightGBM feature importance as representative
lgb_model = lgb.train(lgb_params, lgb.Dataset(X, y), num_boost_round=500)
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': lgb_model.feature_importance(importance_type='gain')
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x='Importance', y='Feature', data=importance.head(15))
plt.title('Top 15 Most Important Features (Feature Gain)', fontsize=15)
plt.xlabel('Importance (Gain)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 5. Submission Generation

We blend the test set predictions and save the submission file matching the hackathon format.

In [ ]:
final_preds = best_w[0] * preds_lgb + best_w[1] * preds_xgb + best_w[2] * preds_cat
final_preds = np.clip(final_preds, 0.0, None)  # Clip to zero since demand cannot be negative

submission = pd.DataFrame({
    'Index': test_idx,
    'demand': final_preds
})

submission.to_csv("results/submission.csv", index=False)
print("Submission file generated successfully!")
print(f"File Shape: {submission.shape}")
print(submission.head())

## 6. Business Value & Real-world Impact for Flipkart

Integrating this predictive pipeline into Flipkart's supply chain unlocks massive business efficiencies:
1. **Logistics Optimization:** Proactively staging delivery fleets in high-congestion regions 30-45 minutes in advance, saving millions in idle time.
2. **Dynamic Slots Management:** Adjusting available delivery windows or applying subtle pricing offsets during gridlock peaks to balance network loads.
3. **Eco-Friendly Routes:** Planning fuel-efficient dispatch schedules by avoiding locations during their predicted peak congestion hours.

This system provides a highly accurate, explainable, and production-ready solution!